# TNG × Faraday rotation: fiducial figures

論文用の基本図を主解析とは別に作る、自己完結したnotebook。
先に `tng_cmb_fr_photoz_specz.ipynb` を実行し、`fr_photoz_outputs/` を残しておく。
本notebookはそのcone・銀河catalog・設定を読み込むだけで、主解析の再実行や変更はしない。

出力: snapshotのガス・磁場投影、密度–磁場分布、近似lightcone、真のzによる銀河mapと $n(z)$、
符号付きRMとRM²、$C_L^{g_i g_i}$、$C_L^{XX}$、$C_L^{Xg_i}$。
$X=R_F^2-\langle R_F^2\rangle$。すべて同じ有限視野から測るfiducial値で、観測forecastではない。
PDF（図中のmap部分はrasterized）と300 dpi PNG、数値CSV/NPZ、設定JSONを保存する。

ガス・磁場図には電子重み付き磁場の既存RM格子ではなく、元のgas snapshotが必要。
初回のみ指定snapshotをchunk単位で読み、その小さな投影・分布cacheを `cache/` に保存する。
依存: numpy, scipy, pandas, matplotlib, h5py, astropy。API key・追加downloadは不要。

In [ ]:
from pathlib import Path
import json, hashlib, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, SymLogNorm
from matplotlib.patches import Polygon
from scipy.ndimage import gaussian_filter
import h5py
from astropy.cosmology import FlatLambdaCDM
from IPython.display import display

## 1. 設定

`MAIN_OUTPUT_DIR` を主解析の出力先へ指定する。TNGデータのmountが変わった場合だけ `BASE_PATH` を設定。
ガス図はsnapshot全体の横幅を保ち、LOSに厚さ10 cMpcの周期slabを切り出す。
`GAS_SLAB_CENTER_FRACTION` で位置、`GAS_SLAB_AXIS` でLOSを変更できる。
理想的な銀河sampleは全保持・photo-z誤差なし・真のcosmological zでビン分け。有限数のshot noiseは残る。

In [ ]:
NOTEBOOK_DIR = (Path.cwd()/'detect-mag'
                if (Path.cwd()/'detect-mag'/'tng_fr_basic_figures.ipynb').is_file()
                else Path.cwd())
MAIN_OUTPUT_DIR = NOTEBOOK_DIR/'fr_photoz_outputs'
OUTPUT_DIR = NOTEBOOK_DIR/'fr_basic_figures'
CACHE_DIR = NOTEBOOK_DIR/'cache'
BASE_PATH = None             # override simulation mount; default: main run_configuration
MAKE_GAS_FIGURES = True      # False: cone/maps/spectra only, no gas snapshot access
ALLOW_SYNTHETIC = False     # validation only; synthetic figures are not physical predictions
GAS_SNAPSHOT = 33            # full snapshot, z ~ 2; independent of cone target bins
GAS_MAP_N = 512
GAS_SLAB_AXIS = 2
GAS_SLAB_CENTER_FRACTION = 0.5
GAS_SLAB_DEPTH_CMPC = 10.0
GAS_READ_BATCH = 250_000
GAS_EXCLUDE_STAR_FORMING = False  # sensitivity option; metadata recorded
PHASE_LOG_NH_EDGES = np.linspace(-8, 4, 121)
PHASE_LOG_B_EDGES = np.linspace(-12, 3, 121) # physical microGauss
MAP_DISPLAY_SMOOTH_ARCMIN = 3.0   # galaxy visualization ONLY, not galaxy spectra
DPI = 300
assert GAS_SLAB_AXIS in (0,1,2) and GAS_MAP_N >= 8
assert 0 <= GAS_SLAB_CENTER_FRACTION < 1 and GAS_SLAB_DEPTH_CMPC > 0
for p in (OUTPUT_DIR,CACHE_DIR): p.mkdir(parents=True,exist_ok=True)
plt.rcParams.update({'font.size':11,'axes.labelsize':12,'savefig.dpi':DPI,
                     'figure.dpi':110,'font.family':'DejaVu Sans','pdf.fonttype':42})
def save_figure(fig,name):
    fig.savefig(OUTPUT_DIR/(name+'.pdf'),bbox_inches='tight')
    fig.savefig(OUTPUT_DIR/(name+'.png'),dpi=DPI,bbox_inches='tight')
    plt.show()
def digest_file(path):
    d=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): d.update(b)
    return d.hexdigest()

## 2. 主解析の出力を読む

RMは全coneの符号付き視線積分を層ごとに保存したもの。**全層を足してから**同じfilterをかけて二乗する。
cone外の磁場や天の川・観測雑音を追加しない。主解析のphoto-z走査や1000 deg²への面積外挿も行わない。

In [ ]:
required=['run_configuration.json','cone_rm_layers.npz','cone_galaxies.csv']
missing=[f for f in required if not (MAIN_OUTPUT_DIR/f).is_file()]
if missing:
    raise FileNotFoundError(f'{MAIN_OUTPUT_DIR}: missing {missing}. Run the main notebook first.')
config=json.loads((MAIN_OUTPUT_DIR/'run_configuration.json').read_text())
if config['MODE']!='tng' and not ALLOW_SYNTHETIC:
    raise ValueError('Synthetic output requires ALLOW_SYNTHETIC=True; it is not TNG data.')
with np.load(MAIN_OUTPUT_DIR/'cone_rm_layers.npz',allow_pickle=False) as f:
    rm_layers=f['rm'].astype(float)
    layer_edges=f['layer_edges'].astype(float)
    fov=float(f['fov_deg'])
    if str(f['mode'].item())!=config['MODE']: raise ValueError('Mixed output modes')
galaxies=pd.read_csv(MAIN_OUTPUT_DIR/'cone_galaxies.csv')
target_edges=np.asarray(config['TRUE_Z_EDGES'],float)
n_target=len(target_edges)-1
n=int(config['MAP_N']); grid_n=int(config['GRID_N'])
assert rm_layers.shape==(len(layer_edges)-1,n,n)
assert np.isclose(fov,config['FOV_DEG'])
assert np.all(np.diff(layer_edges)>0) and np.isfinite(rm_layers).all()
assert np.array_equal(layer_edges[1:-1],target_edges)
assert galaxies[['ix','iy','z_cos']].notna().all().all()
assert ((galaxies[['ix','iy']]>=0)&(galaxies[['ix','iy']]<n)).all().all()
assert ((galaxies.z_cos>=layer_edges[0])&(galaxies.z_cos<layer_edges[-1])).all()
h=float(config['h']); box=float(config['box_mpc'])
cosmo=FlatLambdaCDM(H0=100*h,Om0=float(config['omega_m']))
theta=np.deg2rad(fov); pix_sr=(theta/n)**2; npix=n*n
label_prefix='' if config['MODE']=='tng' else 'SYNTHETIC: '
input_hashes={f:digest_file(MAIN_OUTPUT_DIR/f) for f in required}
print('Source:',MAIN_OUTPUT_DIR,'; simulation:',config['SIMULATION'])
print('Actual patch area:',fov*fov,'deg²; no area extrapolation.')
print('Galaxy z range:',galaxies.z_cos.min(),galaxies.z_cos.max())

## 3. Snapshotガス・磁場分布（streaming + cache）

$B_{\mathrm{phys}}=B_{\mathrm{code}}(h/a^2)\,2.60\times10^{-6}$ G。
ガス質量とDensityの単位は [TNG仕様](https://www.tng-project.org/data/docs/specifications/) に従う。
ガス図はセル中心NGPでslabへ投影。磁場図は同じslabの**質量重み付き**rms強度と平均LOS成分であり、
体積平均磁場でも、RMでもない。

$$
\Sigma_g=\frac{\sum M_g}{A_{\mathrm{physical}}},\qquad
B_{\mathrm{rms},M}=\sqrt{\frac{\sum M_g|\boldsymbol B|^2}{\sum M_g}},\qquad
\overline{B}_{\parallel,M}=\frac{\sum M_g B_\parallel}{\sum M_g}.
$$
密度–磁場分布は**snapshot全体**の質量重み付き分布。固定 $X_H=0.76$ で
$n_H=X_H\rho_{\mathrm{phys}}/m_p$ を定義し、electron densityとは区別する。
star-forming gasは既定で含む。密度/磁場がゼロのセルと表示範囲外の質量は別に集計する。

In [ ]:
MSUN_G=1.98847e33; KPC_CM=3.0856775814913673e21; MP_G=1.67262192369e-24
GAS_CACHE_VERSION='fiducial_slab_mass_B_v1'
def snapshot_files(base,snap):
    return sorted((base/f'snapdir_{snap:03d}').glob(f'snap_{snap:03d}.*.hdf5'),
                  key=lambda p:int(p.name.split('.')[-2]))
def find_gas_base():
    if BASE_PATH is not None: return Path(BASE_PATH).expanduser().resolve()
    candidates=[Path(config['base_path'])]
    for root in [NOTEBOOK_DIR,*list(NOTEBOOK_DIR.parents)[:4],Path.home()]:
        candidates += [root/'sims.TNG'/config['SIMULATION']/'output']
    for p in candidates:
        if (p/f'snapdir_{GAS_SNAPSHOT:03d}').is_dir(): return p.resolve()
    raise FileNotFoundError('Gas snapshot mount not found; set BASE_PATH or MAKE_GAS_FIGURES=False.')

def build_gas_products():
    base=find_gas_base(); files=snapshot_files(base,GAS_SNAPSHOT)
    if not files: raise FileNotFoundError(f'No full snapshot {GAS_SNAPSHOT} at {base}')
    with h5py.File(files[0],'r') as f: header=dict(f['Header'].attrs)
    if len(files)!=int(header['NumFilesPerSnapshot']): raise ValueError('Incomplete snapshot chunks')
    gh=float(header['HubbleParam']); a=float(header['Time']); z=float(header['Redshift'])
    gas_box=float(header['BoxSize'])/(1000*gh)
    if not np.isclose(gas_box,box) or not np.isclose(gh,h):
        raise ValueError('Gas snapshot cosmology/box differs from main cone.')
    assert GAS_SLAB_DEPTH_CMPC<=gas_box and np.isclose(a,1/(1+z))
    meta=dict(version=GAS_CACHE_VERSION,base=str(base),simulation=config['SIMULATION'],
        snap=GAS_SNAPSHOT,n=GAS_MAP_N,axis=GAS_SLAB_AXIS,center=GAS_SLAB_CENTER_FRACTION,
        depth=GAS_SLAB_DEPTH_CMPC,exclude_SF=GAS_EXCLUDE_STAR_FORMING,
        nh_edges=PHASE_LOG_NH_EDGES.tolist(),b_edges=PHASE_LOG_B_EDGES.tolist(),XH=0.76,
        files=[(p.name,p.stat().st_size,p.stat().st_mtime_ns) for p in files])
    key=hashlib.sha256(json.dumps(meta,sort_keys=True).encode()).hexdigest()[:20]
    cache=CACHE_DIR/f'gas_basic_figures_{key}.npz'
    if cache.exists():
        with np.load(cache,allow_pickle=False) as f: result={k:f[k].copy() for k in f.files}
        print('Gas cache hit:',cache.name)
        return result,meta,cache
    mass_map=np.zeros((GAS_MAP_N,GAS_MAP_N)); mb2=mass_map.copy(); mbpar=mass_map.copy()
    phase=np.zeros((len(PHASE_LOG_NH_EDGES)-1,len(PHASE_LOG_B_EDGES)-1))
    totals=np.zeros(4) # selected gas mass, phase-covered mass, invalid/zero nH or B mass, slab mass
    transverse=[k for k in range(3) if k!=GAS_SLAB_AXIS]
    start_time=time.time()
    for file_index,file in enumerate(files):
        with h5py.File(file,'r') as f:
            if 'PartType0' not in f: continue
            g=f['PartType0']
            for name in ['Coordinates','Masses','Density','MagneticField']:
                if name not in g: raise KeyError(f'{file}: missing {name}; full gas snapshot required')
            if GAS_EXCLUDE_STAR_FORMING and 'StarFormationRate' not in g: raise KeyError('Missing SFR')
            for start in range(0,len(g['Masses']),GAS_READ_BATCH):
                sl=slice(start,start+GAS_READ_BATCH)
                pos=g['Coordinates'][sl].astype(float)/(1000*gh)
                mass=g['Masses'][sl].astype(float)*1e10/gh
                B=g['MagneticField'][sl].astype(float)*(gh/a**2)*2.60 # microGauss
                rho=g['Density'][sl].astype(float)*(1e10*MSUN_G/gh)/(a*KPC_CM/gh)**3
                good=np.isfinite(pos).all(1)&np.isfinite(mass)&(mass>0)&np.isfinite(B).all(1)
                if GAS_EXCLUDE_STAR_FORMING: good &= g['StarFormationRate'][sl]<=0
                pos,mass,B,rho=pos[good],mass[good],B[good],rho[good]
                bmag=np.linalg.norm(B,axis=1); nh=0.76*rho/MP_G
                totals[0]+=mass.sum()
                valid=(nh>0)&np.isfinite(nh)&(bmag>0)
                totals[2]+=mass[~valid].sum()
                hist=np.histogram2d(np.log10(nh[valid]),np.log10(bmag[valid]),
                    bins=[PHASE_LOG_NH_EDGES,PHASE_LOG_B_EDGES],weights=mass[valid])[0]
                phase+=hist; totals[1]+=hist.sum()
                distance=(pos[:,GAS_SLAB_AXIS]-GAS_SLAB_CENTER_FRACTION*gas_box+gas_box/2)%gas_box-gas_box/2
                keep=abs(distance)<GAS_SLAB_DEPTH_CMPC/2
                xy=pos[keep][:,transverse]%gas_box
                idx=np.floor(xy/gas_box*GAS_MAP_N).astype(int)
                flat=idx[:,0]*GAS_MAP_N+idx[:,1]
                weights=[mass[keep],mass[keep]*bmag[keep]**2,mass[keep]*B[keep,GAS_SLAB_AXIS]]
                for out,w in zip([mass_map,mb2,mbpar],weights):
                    out+=np.bincount(flat,weights=w,minlength=GAS_MAP_N**2).reshape(out.shape)
                totals[3]+=mass[keep].sum()
        print(f'Gas chunks {file_index+1}/{len(files)}; elapsed {(time.time()-start_time)/60:.1f} min')
    if mass_map.sum()<=0 or totals[0]<=0: raise ValueError('No usable gas in snapshot/slab')
    assert np.isclose(mass_map.sum(),totals[3])
    result=dict(mass=mass_map,mb2=mb2,mbpar=mbpar,phase=phase,totals=totals,
                a=np.array(a),z=np.array(z),box=np.array(gas_box))
    # Replace only a completed temporary cache; an interrupted run never becomes a cache hit.
    temp=cache.with_suffix('.tmp.npz');np.savez_compressed(temp,**result);temp.replace(cache)
    return result,meta,cache

gas_metadata=None; gas_cache=None
if MAKE_GAS_FIGURES:
    if config['MODE']!='tng': raise ValueError('Synthetic cone has no physical gas snapshot.')
    gas,gas_metadata,gas_cache=build_gas_products()
    a=float(gas['a']); m=gas['mass']; extent=[0,float(gas['box'])]*2
    area_pc2=(float(gas['box'])/GAS_MAP_N*a*1e6)**2
    sigma=m/area_pc2
    brms=np.sqrt(np.divide(gas['mb2'],m,out=np.full_like(m,np.nan),where=m>0))
    bpar=np.divide(gas['mbpar'],m,out=np.full_like(m,np.nan),where=m>0)
    fig,axes=plt.subplots(1,3,figsize=(15,4.6),constrained_layout=True)
    for ax,field,title,unit,cmap in zip(axes,[sigma,brms,bpar],
        [r'Gas surface density',r'Mass-weighted $B_{\mathrm{rms}}$',r'Mass-weighted $B_\parallel$'],
        [r'$M_\odot\,\mathrm{pc}^{-2}$',r'$\mu\mathrm{G}$',r'$\mu\mathrm{G}$'],['magma','viridis','RdBu_r']):
        if field is bpar:
            vmax=max(float(np.nanpercentile(abs(field),99)),1e-20)
            norm=SymLogNorm(linthresh=vmax*0.01,vmin=-vmax,vmax=vmax)
        else:
            positive=field[np.isfinite(field)&(field>0)]
            if not len(positive): raise ValueError('No positive values for log gas/B map')
            lo,hi=np.percentile(positive,[1,99]);hi=max(hi,lo*1.01)
            norm=LogNorm(vmin=lo,vmax=hi)
        im=ax.imshow(np.ma.masked_invalid(np.where(m>0,field,np.nan)).T,origin='lower',
                     extent=extent,cmap=cmap,norm=norm,rasterized=True)
        ax.set(title=title,xlabel='Transverse distance [cMpc]',ylabel='Transverse distance [cMpc]')
        fig.colorbar(im,ax=ax,label=unit,shrink=0.82)
    fig.suptitle(f"{config['SIMULATION']}, z={float(gas['z']):.3f}; {GAS_SLAB_DEPTH_CMPC:g} cMpc slab; NGP")
    save_figure(fig,'01_gas_magnetic_slab')
    dnh=np.diff(PHASE_LOG_NH_EDGES); db=np.diff(PHASE_LOG_B_EDGES)
    pdf=gas['phase']/gas['totals'][0]/(dnh[:,None]*db[None,:])
    positive=pdf[pdf>0]
    if not len(positive): raise ValueError('Density/B plotting range contains no mass; widen edges.')
    fig,ax=plt.subplots(figsize=(6.7,5.0),constrained_layout=True)
    im=ax.pcolormesh(PHASE_LOG_NH_EDGES,PHASE_LOG_B_EDGES,np.ma.masked_less_equal(pdf.T,0),
        norm=LogNorm(vmin=max(positive.min(),positive.max()*1e-6),vmax=positive.max()),
        cmap='magma',rasterized=True)
    ax.set(xlabel=r'$\log_{10}(n_H/\mathrm{cm}^{-3})$',ylabel=r'$\log_{10}(|B|/\mu\mathrm{G})$',
           title=f"{config['SIMULATION']}, z={float(gas['z']):.3f}; full snapshot")
    fig.colorbar(im,ax=ax,label='Gas mass fraction per dex²')
    save_figure(fig,'02_density_magnetic_distribution')
    print('Phase plot mass coverage:',gas['totals'][1]/gas['totals'][0])
    print('Invalid/zero nH or B mass fraction:',gas['totals'][2]/gas['totals'][0])
    np.savez_compressed(OUTPUT_DIR/'gas_figure_data.npz',**gas,nh_edges=PHASE_LOG_NH_EDGES,b_edges=PHASE_LOG_B_EDGES)

## 4. 近似lightconeと銀河の赤方偏移分布

主解析のtile境界・nearest snapshot割当てを表示。箱の周期複製と回転を用いた幾何学近似であり、
独立な広域宇宙ではない。保存catalogは角度pixelのみなので、点の横位置はpixel中心から再構成する。
3Dで正確な各銀河位置を復元した図ではない。全銀河を表示し、表示の間引きはしない。

In [ ]:
zgrid=np.linspace(0,layer_edges[-1]+0.2,10001)
chigrid=cosmo.comoving_distance(zgrid).value
chi=np.interp(galaxies.z_cos,zgrid,chigrid)
theta_x=(galaxies.ix.to_numpy()+0.5)/n*theta-theta/2
x=theta_x*chi
tiles=config['tiles']; palette=plt.get_cmap('tab20')
fig,axes=plt.subplots(1,2,figsize=(13,4.8),constrained_layout=True)
ax=axes[0]
for j,t in enumerate(tiles):
    lo,hi=t['chi_lo'],t['chi_hi']; half=theta/2
    poly=Polygon([[lo,-lo*half],[hi,-hi*half],[hi,hi*half],[lo,lo*half]],
                  facecolor=palette(j%20),alpha=0.18,edgecolor='0.6',linewidth=0.6)
    ax.add_patch(poly)
    ax.text((lo+hi)/2,hi*half+10,f"{t['snap']}",rotation=90,fontsize=7,ha='center')
sc=ax.scatter(chi,x,c=galaxies.z_cos,s=1,cmap='viridis',rasterized=True)
for z in target_edges:
    ax.axvline(np.interp(z,zgrid,chigrid),ls='--',color='0.2',lw=0.8)
ax.set(xlabel=r'LOS comoving distance $\chi$ [cMpc]',ylabel='Transverse x [cMpc]',
       title=label_prefix+'Approximate cone; labels: snapshot IDs')
ax.set_xlim(tiles[0]['chi_lo'],tiles[-1]['chi_hi'])
ax.set_ylim(-tiles[-1]['chi_hi']*theta/2-10,tiles[-1]['chi_hi']*theta/2+75)
sec=ax.secondary_xaxis('top',functions=(lambda c:np.interp(c,chigrid,zgrid),
                                      lambda z:np.interp(z,zgrid,chigrid)))
sec.set_xlabel('Cosmological redshift')
fig.colorbar(sc,ax=ax,label='True redshift',shrink=0.8)
z_edges=np.linspace(layer_edges[0],layer_edges[-1],61)
nz,_=np.histogram(galaxies.z_cos,bins=z_edges)
nz_density=nz/(fov**2*np.diff(z_edges))
axes[1].step(z_edges,np.r_[nz_density,nz_density[-1]],where='post',color='black')
for i,(lo,hi) in enumerate(zip(target_edges[:-1],target_edges[1:])):
    axes[1].axvspan(lo,hi,color=palette(i*2),alpha=0.16)
axes[1].set(xlabel='True cosmological redshift',ylabel=r'$dN/(dz\,d\Omega)$ [deg$^{-2}$]',
            title=label_prefix+'Parent galaxy distribution; no photo-z errors')
save_figure(fig,'03_lightcone_and_nz')
pd.DataFrame(dict(z_lo=z_edges[:-1],z_hi=z_edges[1:],count=nz,
    dN_dz_deg2=nz/(fov**2*np.diff(z_edges)))).to_csv(OUTPUT_DIR/'galaxy_nz.csv',index=False)
pd.DataFrame([{k:v for k,v in t.items() if k not in ('perm','signs','offset')} for t in tiles]).to_csv(OUTPUT_DIR/'lightcone_tiles.csv',index=False)

## 5. 理想的な銀河・RM map

銀河mapは真のzで分類し、$g_i=N_i/\bar N_i-1$。スペクトルには平滑化前の銀河mapを使う。
見やすくするためmap図だけ3分角で平滑化し、そのことを図に明記する。
RMには主解析と同じGaussian filterを適用し、$X=R_F^2-\langle R_F^2\rangle$ を作る。

In [ ]:
lx=2*np.pi*np.fft.fftfreq(n,d=theta/n)
LX,LY=np.meshgrid(lx,lx,indexing='ij'); LMAG=np.hypot(LX,LY)
rm_fwhm=float(config['RM_SMOOTH_FWHM_ARCMIN'])
sigma_rm=np.deg2rad(rm_fwhm/60)/np.sqrt(8*np.log(2))
rm_filter=np.exp(-0.5*LMAG**2*sigma_rm**2)
R=np.fft.ifft2(np.fft.fft2(rm_layers.sum(0))*rm_filter).real
X=R**2-np.mean(R**2)
gmaps=[];counts=[]
for lo,hi in zip(target_edges[:-1],target_edges[1:]):
    keep=(galaxies.z_cos>=lo)&(galaxies.z_cos<hi)
    flat=galaxies.loc[keep,'ix'].to_numpy(dtype=int)*n+galaxies.loc[keep,'iy'].to_numpy(dtype=int)
    cnt=np.bincount(flat,minlength=npix).reshape(n,n)
    if not cnt.sum(): raise ValueError(f'Empty true-z bin {lo}-{hi}')
    gmaps.append(cnt/cnt.mean()-1);counts.append(int(cnt.sum()))
gmaps=np.array(gmaps);counts=np.array(counts)
fig,axes=plt.subplots(1,n_target,figsize=(5*n_target,4.6),squeeze=False,constrained_layout=True)
sigma_pix=(MAP_DISPLAY_SMOOTH_ARCMIN/np.sqrt(8*np.log(2)))/(fov*60/n)
for i,ax in enumerate(axes[0]):
    view=gaussian_filter(gmaps[i],sigma_pix,mode='wrap') if sigma_pix>0 else gmaps[i]
    im=ax.imshow(view.T,origin='lower',extent=[-fov/2,fov/2]*2,cmap='viridis',rasterized=True)
    ax.set(xlabel='Angle x [deg]',ylabel='Angle y [deg]',
        title=f'{target_edges[i]:g}<z<{target_edges[i+1]:g}; N={counts[i]}')
    fig.colorbar(im,ax=ax,label=r'$\delta_g$ (display-smoothed)',shrink=0.8)
fig.suptitle(label_prefix+f'Fiducial galaxies; display FWHM={MAP_DISPLAY_SMOOTH_ARCMIN:g} arcmin')
save_figure(fig,'04_fiducial_galaxy_maps')
fig,axes=plt.subplots(1,2,figsize=(10,4.6),constrained_layout=True)
for ax,field,title,unit in zip(axes,[R,X],['Signed filtered RM','Centered filtered RM²'],
                             ['rad m$^{-2}$','rad² m$^{-4}$']):
    vmax=max(float(np.max(abs(field))),1e-30)
    im=ax.imshow(field.T,origin='lower',extent=[-fov/2,fov/2]*2,cmap='RdBu_r',
        norm=SymLogNorm(linthresh=vmax*0.01,vmin=-vmax,vmax=vmax),rasterized=True)
    ax.set(title=title,xlabel='Angle x [deg]',ylabel='Angle y [deg]')
    fig.colorbar(im,ax=ax,label=unit,shrink=0.8)
fig.suptitle(label_prefix+f'Full cone {layer_edges[0]:g}<z<{layer_edges[-1]:g}; RM FWHM={rm_fwhm:g} arcmin')
save_figure(fig,'05_fiducial_rm_maps')
np.savez_compressed(OUTPUT_DIR/'fiducial_maps.npz',R=R,X=X,gmaps=gmaps,counts=counts,
                    target_edges=target_edges,layer_edges=layer_edges,fov_deg=fov,rm_fwhm_arcmin=rm_fwhm)

## 6. Fiducial角度パワースペクトル

主解析と同じFFT規格化・角度cutを使う。ゼロmodeは除外。
$$
C_L^{ab}=\frac{\Omega_{\mathrm{pix}}}{N_{\mathrm{pix}}}
\operatorname{Re}(F_aF_b^*)\quad\text{（annular average）}.
$$
$C_L^{gg}$ の実線は有限catalogのshot noiseを含み、点線は $N_L^{gg}=1/\bar n_{\mathrm{sr}}$
を引いた診断値。点線が負になっても隠さない。
$C_L^{XX}$ は全coneのRM²のautoであり、各z層のautoではない。
$C_L^{Xg_i}$ は全coneのRM²と真のzで選択した銀河とのcross。
有限視野のcrossは負にもなり得るのでsymlogで描く。負を絶対値に変えない。

各bandのmode数をCSVに保存するが、単一の周期箱coneから厳密なensemble平均や誤差を主張しない。
実サーベイのmask・window coupling・connected covarianceはここでは計算しない。

In [ ]:
ell_min=2*np.pi/theta
chi_min=float(cosmo.comoving_distance(float(layer_edges[0])).value)
grid_limit=0.5*np.pi*chi_min/(box/grid_n)
ell_limit=min(float(config['ELL_MAX']),0.65*np.pi*n/theta,grid_limit)
if ell_limit<=1.5*ell_min: raise ValueError('Insufficient resolved modes')
band_edges=np.geomspace(ell_min*0.99,ell_limit,int(config['N_BANDS'])+1)
bands=[(lo,hi,(LMAG>=lo)&(LMAG<hi)) for lo,hi in zip(band_edges[:-1],band_edges[1:])]
bands=[(lo,hi,m) for lo,hi,m in bands if m.sum()>=4]
ell=np.array([LMAG[m].mean() for lo,hi,m in bands])
def spectrum(a,b):
    power=pix_sr/npix*(np.fft.fft2(a)*np.fft.fft2(b).conj()).real
    return np.array([power[m].mean() for lo,hi,m in bands])
cxx=spectrum(X,X)
cgg=np.array([spectrum(g,g) for g in gmaps])
cxg=np.array([spectrum(X,g) for g in gmaps])
shot=theta**2/counts
assert np.all(cxx>=0) and np.all(cgg>=0)
assert np.all(cxg**2<=cxx[None,:]*cgg*(1+1e-10)+1e-30)
fig,axes=plt.subplots(1,3,figsize=(15,4.6),constrained_layout=True)
colors=plt.get_cmap('tab10')
for i in range(n_target):
    label=f'{target_edges[i]:g}<z<{target_edges[i+1]:g}'
    axes[0].plot(ell,cgg[i],'-o',color=colors(i),label=label,ms=4)
    axes[0].plot(ell,cgg[i]-shot[i],':',color=colors(i),lw=1.5)
    axes[2].plot(ell,cxg[i],'-o',color=colors(i),label=label,ms=4)
axes[1].plot(ell,cxx,'-o',color='black',ms=4)
axes[1].set_yscale('log')
for ax,values in [(axes[0],np.r_[cgg.ravel(),(cgg-shot[:,None]).ravel()]),(axes[2],cxg.ravel())]:
    ax.set_yscale('symlog',linthresh=max(np.max(abs(values))*0.01,1e-30))
    ax.axhline(0,color='0.6',lw=0.6);ax.legend(fontsize=9)
for ax in axes: ax.set_xscale('log');ax.set_xlabel(r'Multipole $L$')
axes[0].text(0.03,0.03,'Solid: raw; dotted: shot-subtracted',transform=axes[0].transAxes,fontsize=8,
             bbox=dict(facecolor='white',alpha=0.8,edgecolor='none'))
axes[0].set(title=r'$C_L^{g_i g_i}$',ylabel=r'$C_L$ [sr]')
axes[1].set(title=r'$C_L^{XX}$, full cone',ylabel=r'$C_L$ [rad$^4$ m$^{-8}$ sr]')
axes[2].set(title=r'$C_L^{Xg_i}$',ylabel=r'$C_L$ [rad$^2$ m$^{-4}$ sr]')
fig.suptitle(label_prefix+f'Fiducial spectra; {fov*fov:g} deg² patch; no observational noise')
save_figure(fig,'06_fiducial_angular_spectra')
rows=[]
for i in range(n_target):
    for b,(lo,hi,m) in enumerate(bands):
        rows.append(dict(z_lo=target_edges[i],z_hi=target_edges[i+1],L=ell[b],L_lo=lo,L_hi=hi,
            patch_modes=int(m.sum()),N_gal=counts[i],n_deg2=counts[i]/fov**2,
            C_gg_sr=cgg[i,b],N_gg_sr=shot[i],C_gg_minus_shot_sr=cgg[i,b]-shot[i],
            C_XX_rad4_m8_sr=cxx[b],C_Xg_rad2_m4_sr=cxg[i,b],
            r_Xg=cxg[i,b]/np.sqrt(cxx[b]*cgg[i,b]) if cxx[b]*cgg[i,b]>0 else np.nan))
spectra_table=pd.DataFrame(rows)
spectra_table.to_csv(OUTPUT_DIR/'fiducial_spectra.csv',index=False)
display(spectra_table.head())

## 7. 保存・論文での解釈

図はfiducialな一視野の構造と相関を示すためのもの。格子・slab厚・snapshot cadence・seed・TNG解像度を変え、
構造やスペクトルが安定かを確認してから論文の物理的結論に用いる。
磁場の質量重み付き投影、密度–磁場の質量分布、electron-weighted RMは異なる量なのでcaptionで区別する。
ガス分布図は指定snapshot、ライトコーンとスペクトルは主解析の複数snapshotの構成である。
syntheticを許可した場合は図のタイトルに明示される。主解析の出力・cacheは上書きしない。

In [ ]:
assert np.isclose(X.mean(),0,atol=1e-10*max(1.,np.std(X)))
assert np.allclose(gmaps.mean(axis=(1,2)),0,atol=1e-12)
provenance=dict(source_directory=str(MAIN_OUTPUT_DIR.resolve()),source_hashes=input_hashes,
    main_configuration=config,status='fiducial finite-patch figures; not an observational forecast',
    galaxy_selection='true cosmological z; full retained parent sample; no photo-z scatter',
    spectrum_galaxy_smoothing='none',display_smoothing_arcmin=MAP_DISPLAY_SMOOTH_ARCMIN,
    gas_cache=str(gas_cache) if gas_cache is not None else None,gas_metadata=gas_metadata,
    gas_projection='NGP slab; mass-weighted B rms and signed LOS mean',
    shot_noise='1/n_sr; raw and subtracted gg both exported',actual_patch_area_deg2=fov*fov,
    versions={k:__import__(k).__version__ for k in ['numpy','scipy','pandas','matplotlib','h5py','astropy']})
(OUTPUT_DIR/'figure_configuration.json').write_text(json.dumps(provenance,indent=2),encoding='utf-8')
print('Figures and data:',OUTPUT_DIR.resolve())
display(pd.DataFrame([dict(file=p.name,KiB=round(p.stat().st_size/1024,1))
                      for p in sorted(OUTPUT_DIR.iterdir()) if p.is_file()]))